# <p style="font-size:30px;text-align:center"><b>Lab: Import and Run a Git-Backed Notebook in Databricks</b></p>

## Overview

In this lab, you will import a Git-backed notebook into Databricks by cloning a GitHub repository, attach a compute resource, and run a PySpark pipeline that loads, transforms, and aggregates a sales dataset. Along the way, you will make a small code change and then reflect on what it means to save a notebook versus committing that change back to Git.

## Learning Objectives

By the end of this lab, you will be able to:

- Clone a GitHub repository into the Databricks Repos / Git Folders area.
- Open and run a notebook sourced from a Git repository.
- Load a CSV file from a repo path using PySpark.
- Apply filters, transformations, and aggregations to a DataFrame.
- Write aggregated results to a CSV file.
- Explain the difference between saving a notebook locally and committing a change to Git.

### Prerequisites

- A Databricks or Azure Databricks workspace with at least one active cluster available.
- A GitHub account and a personal access token (PAT) with `repo` scope, or SSH key configured.
- A GitHub repository that you can use for this lab.
- `orders.csv` uploaded to your GitHub repository before starting (schema described in the **Data** section below).

---

## Essential Terms and Concepts

### Git-Backed Notebooks

By default, Databricks notebooks live in the **Workspace** - a managed file system internal to Databricks. They are easy to create and share, but they have no native version history. If you overwrite a cell and save, the previous version is gone.

**Git-backed notebooks** are notebooks that live inside a cloned Git repository. Databricks surfaces this through the **Repos** area (older workspaces) or **Git Folders** (newer workspaces). Both refer to the same capability: Databricks checks out a repository from a remote provider - GitHub, GitLab, Bitbucket, and others - and makes its contents available as a browseable folder. You can open and run notebooks directly from that folder just as you would from the Workspace.

The key difference is that any edit you make is a local file change inside a Git working tree. Databricks will not automatically push those changes anywhere; you have to explicitly commit and push, just as you would from a terminal or an IDE.

### Repos Path vs. Workspace Path

Notebooks opened from the Workspace are stored at paths like `/Users/<your-email>/notebook_name`. Notebooks opened from a cloned repo are stored at paths like `/Repos/<your-email>/<repo-name>/notebook_name`. This matters when your notebook needs to reference other files - including data files - that live in the same repository. You will use this path pattern in the data-loading step below.

### Saving vs. Committing

These two actions are often confused when working in Databricks:

- **Saving** a notebook writes your changes to Databricks' internal state. In a Git-backed notebook, this updates the local file on the Databricks file system , the same way editing a file in a text editor and saving it touches the file on disk, without doing anything to Git.
- **Committing** packages your saved changes into a Git commit and records them in the repository's history. Until you commit, your edits exist only locally and are invisible to anyone else working from the same repo.

In a team environment, not committing is equivalent to not sharing your work.

---

## Data

### orders.csv

The file will be shared with you on the learning platform. Upload it to your GitHub repository if it already exists, or upload it later when you create the repository in this lab. Place it in a `Data/` folder at the root of your repo so that the file path used in this notebook resolves correctly.

| Column | Data Type | Description |
|---|---|---|
| `ord_no` | INTEGER | Unique order number |
| `purch_amt` | FLOAT | Total purchase amount for the order |
| `ord_date` | DATE | Date the order was placed |
| `customer_id` | INTEGER | Identifier for the customer |
| `salesman_id` | INTEGER | Identifier for the salesperson who handled the order |

---

## Step 1 - Connect Databricks to GitHub

Before you can clone a repository, Databricks needs credentials to authenticate with GitHub.

1. In the Databricks sidebar, click your **username** in the top right corner and select **Settings**.
2. Navigate to **Linked accounts** (or **Git integration**, depending on your workspace version).
3. Click on **Add Git credential**, and Under **Git provider**, select **GitHub**.
4. Select **Personal access token**.
4. Enter your GitHub **username** and paste your **personal access token** (PAT). If you do not have a PAT, generate one at [github.com/settings/tokens](https://github.com/settings/tokens) with the `repo` scope enabled.
5. Click **Save**.

> **Note:** If your workspace uses Azure Active Directory SSO and your organisation has already connected GitHub at the workspace level, you may not need to configure credentials manually. Check with your workspace administrator.

---

## Step 2 - Clone Your Repository

1. In the Databricks sidebar, click **Workspace**.
2. Navigate to **Repos** or **Git Folders** (the label depends on your workspace version - both refer to the same feature).
3. Click **Add/ Create repo**. In the dialog box that opens up, paste your repository URL if you want to clone from an existing repo: `https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git`.
    - You can uncheck the **Create repo by cloning a Git repository** option if you don't want to clone from an existing repo and create an entirely new one. 
 4. Enter a name for your repo, and leave the **Git provider** set to **GitHub** and click **Create repo** (or **Clone**, depending on your UI version).

Databricks will clone the repository and make it available as a folder under `/Repos/<your-email>/YOUR_REPO_NAME/`. You should see its contents - including this notebook and the `data/` folder - in the file tree.

> **Tip:** If the clone fails with an authentication error, double-check that your PAT has the `repo` scope and that you saved your credentials in Step 1.

---

## Step 3 - Attach Compute and Open This Notebook

1. From the repo folder in the file tree, open this notebook by clicking on it.
2. In the top-right corner of the notebook, click the **Connect** dropdown and select your cluster.
3. Wait for the cluster state to show as **Connected** (the indicator turns green).

Once connected, you are ready to run cells. Proceed through the remaining steps by running each code cell in order.

---

## Step 4 - Load the Dataset

The cell below constructs the path to `orders.csv` using the standard Databricks Repos path pattern. It then reads the file into a PySpark DataFrame with schema inference enabled and displays the first few rows.

In [0]:
import os

# Build the path to the data file relative to this notebook's location in the repo.
# os.path.dirname(__file__) resolves to the directory containing this notebook.
# Adjust the path below if you placed orders.csv in a different folder.
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
data_path = os.path.join(notebook_dir, 'Data', 'orders.csv')

print(f"Loading data from: {data_path}")

In [0]:
orders_df = (
    spark
    .read
    .option('header', True)
    .option('inferSchema', True)
    .csv(f'file:{data_path}')
)

display(orders_df)

Now inspect the inferred schema. Confirm that `purch_amt` has been read as a numeric type and `ord_date` as a string (PySpark infers dates as strings by default unless told otherwise).

In [0]:
orders_df.printSchema()

---

## Step 5 - Filter and Transform

Filter the dataset to keep only high-value orders - those where the purchase amount exceeds **500**. Then add a derived column, `order_tier`, that labels each remaining order as `'Premium'` if `purch_amt` is above **1500**, and `'Standard'` otherwise.

In [0]:
from pyspark.sql import functions as F

# Filter: keep only orders with a purchase amount greater than 500.
filtered_df = orders_df.filter(F.col('purch_amt') > 1000)

# Transform: classify each order into a tier based on purchase amount.
transformed_df = filtered_df.withColumn(
    'order_tier',
    F.when(F.col('purch_amt') > 1500, 'Premium').otherwise('Standard')
)

display(transformed_df)

---

## Step 6 - Aggregate and Save Results

Aggregate the filtered and transformed data to produce a summary of sales performance per salesperson. The output includes:

- `total_orders` - number of qualifying orders handled.
- `total_revenue` - sum of purchase amounts, rounded to two decimal places.
- `avg_order_value` - average purchase amount per order, rounded to two decimal places.

In [0]:
summary_df = (
    transformed_df
    .groupBy('salesman_id')
    .agg(
        F.count("ord_no").alias('total_orders'),
        F.round(F.sum('purch_amt'), 2).alias('total_revenue'),
        F.round(F.avg('purch_amt'), 2).alias('avg_order_value')
    )
    .orderBy('total_revenue', ascending=False)
)

display(summary_df)

Write the aggregated summary to a CSV file. The `coalesce(1)` call consolidates the output into a single file rather than one file per Spark partition. The result is saved to a `output/` folder relative to this notebook.

In [0]:
output_path = os.path.join(notebook_dir, 'output', 'sales_summary')

(
    summary_df
    .coalesce(1)
    .write
    .mode('overwrite')
    .option('header', True)
    .csv(f"file:{output_path}")
)

print(f"Summary written to: {output_path}")

---

## Step 7 - Exercise: Change the Filter Condition

Go back to the filter cell in **Step 5** and change the purchase amount threshold from **500** to **1000**:

```python
# Before
filtered_df = orders_df.filter(F.col('purch_amt') > 500)

# After
filtered_df = orders_df.filter(F.col('purch_amt') > 1000)
```

Then re-run all cells from Step 5 onwards (**Run** → **Run all below**) and observe how the aggregation results change.

> **Questions to consider:** How many orders are excluded by the stricter threshold? Does the ranking of salespeople by total revenue change?

---

## Step 8 - Saving vs. Committing

After making the edit above, your notebook has been saved automatically by Databricks , but that does not mean Git knows about your change. To commit the changes:

1. Click the **Git** button at the top of the notebook.
2. Review the diff, and enter a commit message.
3. Click **Commit & Push**. Only after this step will your teammates - or your own future self pulling the repo from another machine - see the change.

---

## Congratulations!

You have cloned a Git repository into Databricks, run a PySpark pipeline end to end - loading, filtering, transforming, aggregating, and writing results - and experienced first-hand the difference between saving a notebook and committing a change to Git. This Git-backed development pattern is how data engineering teams collaborate on notebooks in practice.